# Gradient Accumulation Training Loop in PyTorch

**Task:** Using PyTorch, write a function that takes a custom loss function and a model, then trains the model for a specified number of epochs on a given `DataLoader`. The function must support gradient accumulation over N batches and log the average loss every M steps.

## Background: gradient accumulation

Gradient accumulation lets you train with a large effective batch size when the batch won't fit in GPU memory.

### The problem it solves

Memory use in training scales with batch size, because activations for the whole batch are stored for backprop. Sometimes you want a large batch (say 256) for stable gradients or to match a paper's recipe, but only 32 samples fit on your GPU.

### What it does

Instead of one big batch, you run N small batches (micro-batches), summing their gradients, and call `optimizer.step()` once after N of them:

```
effective batch size = batch_size × accum_steps
e.g. 32 × 8 = 256
```

The math works because the gradient of a mean loss over 256 samples is the average of the gradients from 8 groups of 32. Since `.backward()` adds into `.grad`, you just delay the optimizer step and divide each loss by N so the sum becomes a mean.

### When it's useful

- Large models (LLMs, big vision models) where even small batches nearly fill memory.
- Reproducing a result that specifies a large batch size on smaller hardware.
- Tasks where small batches give noisy gradients, such as contrastive learning or noisy-label training.

### Trade-offs

- **It saves memory, not time.** You still do the same number of forward and backward passes, so it's no faster, and often slightly slower than a true large batch.
- **BatchNorm doesn't behave identically.** Its statistics are computed per micro-batch, not over the full effective batch. LayerNorm and GroupNorm don't have this issue.
- **Batch-dependent losses** (e.g. in-batch negatives in contrastive learning) see only the micro-batch, so accumulation isn't mathematically equivalent there.
- **Learning rate and scheduler steps** should be counted in optimizer steps, not micro-batches, which is why the code steps the scheduler once per update.

If you're training on a single GPU with limited memory, it's usually the first thing to reach for, along with mixed precision and gradient checkpointing.

## Algorithm (pseudocode)

```text
for each epoch:
    pending = 0                          # batches accumulated since last optimizer step
    for each (inputs, targets) in dataloader:
        move batch to device
        outputs = model(inputs)
        loss    = loss_fn(outputs, targets)

        (loss / accum_steps).backward()  # grads ADD into .grad across batches

        window_loss    += loss.item()    # unscaled loss, for logging
        window_batches += 1
        pending        += 1

        if pending == accum_steps:
            optimizer_step(pending)
            pending = 0

    if pending > 0:                      # leftover batches at epoch end
        optimizer_step(pending)


def optimizer_step(pending):
    1. if pending < accum_steps:
        multiply grads by accum_steps / pending   # fix under-scaled short group
    2. optionally clip gradient norm
    3. optimizer.step()
    4. optionally scheduler.step()
    5. optimizer.zero_grad()
    6. global_step += 1
    7. if global_step % log_every == 0:
        avg = window_loss / window_batches
        log + append (global_step, avg) to history
        reset window_loss and window_batches
```

## Imports

In [1]:
import logging
from typing import Callable, Optional

import torch
from torch import nn
from torch.utils.data import DataLoader

logger = logging.getLogger(__name__)

## The `train` function

Notes:

- A "step" is one optimizer update.
- The logged loss is the mean of the **unscaled** per-batch losses seen since the previous log.
- If an epoch ends with fewer than `accum_steps` batches pending, they are flushed as one optimizer step with gradients rescaled to a proper mean.

In [2]:
def train(
    model: nn.Module,
    loss_fn: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    epochs: int,
    accum_steps: int = 1,
    log_every: int = 100,
    device: Optional[torch.device | str] = None,
    max_grad_norm: Optional[float] = None,
    scheduler: Optional[torch.optim.lr_scheduler.LRScheduler] = None,
) -> list[tuple[int, float]]:
    """Train `model` with a custom loss, gradient accumulation and periodic logging.

    Args:
        model: The model to train.
        loss_fn: Callable (outputs, targets) -> scalar loss tensor.
        dataloader: Yields (inputs, targets) batches.
        optimizer: Optimizer for the model's parameters.
        epochs: Number of passes over the dataloader.
        accum_steps: Number of batches to accumulate gradients over before each
            optimizer step (N). Effective batch size = batch_size * accum_steps.
        log_every: Log the average loss every this many optimizer steps (M).
        device: Device to train on. Defaults to the model's current device.
        max_grad_norm: If set, clip the global gradient norm to this value.
        scheduler: Optional LR scheduler, stepped once per optimizer step.

    Returns:
        History as a list of (optimizer_step, average_loss) tuples, one per log.

    Notes:
        A "step" is one optimizer update. The logged loss is the mean of the
        unscaled per-batch losses seen since the previous log. If an epoch ends
        with fewer than `accum_steps` batches pending, they are flushed as one
        optimizer step with gradients rescaled to a proper mean.
    """
    if accum_steps < 1 or log_every < 1 or epochs < 1:
        raise ValueError("epochs, accum_steps and log_every must be >= 1")

    if device is None:
        device = next(model.parameters()).device
    device = torch.device(device)
    model.to(device)
    model.train()

    global_step = 0          # optimizer steps taken
    window_loss = 0.0        # sum of per-batch losses since last log
    window_batches = 0       # number of batches since last log
    history: list[tuple[int, float]] = []

    def optimizer_step(pending: int) -> None:
        nonlocal global_step, window_loss, window_batches
        # Each batch was scaled by 1/accum_steps; fix up a short final group.
        if pending != accum_steps:
            scale = accum_steps / pending
            for p in model.parameters():
                if p.grad is not None:
                    p.grad.mul_(scale)
        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

        if global_step % log_every == 0:
            avg = window_loss / window_batches
            history.append((global_step, avg))
            logger.info("step %d | avg loss %.6f", global_step, avg)
            window_loss, window_batches = 0.0, 0

    optimizer.zero_grad(set_to_none=True)

    for epoch in range(epochs):
        pending = 0
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)

            # Scale so the accumulated gradient equals the mean over N batches.
            (loss / accum_steps).backward()

            window_loss += loss.item()
            window_batches += 1
            pending += 1

            if pending == accum_steps:
                optimizer_step(pending)
                pending = 0

        if pending > 0:  # flush leftover batches at epoch end
            optimizer_step(pending)

        logger.info("epoch %d/%d complete", epoch + 1, epochs)

    return history

## Demo

Linear regression on synthetic data with a custom Huber-style loss (`smooth_l1_loss`).
1000 samples at `batch_size=32` gives 32 batches per epoch, so with `accum_steps=4` each epoch is 8 optimizer steps. The effective batch size is 128.

In [3]:
from torch.utils.data import TensorDataset

# force=True so logging config takes effect inside Jupyter, which pre-installs handlers
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
torch.manual_seed(0)

X = torch.randn(1000, 10)
y = X @ torch.randn(10, 1) + 0.1 * torch.randn(1000, 1)
loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

net = nn.Linear(10, 1)
huber = lambda out, tgt: nn.functional.smooth_l1_loss(out, tgt)  # custom loss
opt = torch.optim.SGD(net.parameters(), lr=0.05)

hist = train(net, huber, loader, opt, epochs=3, accum_steps=4, log_every=5)
assert hist[-1][1] < hist[0][1], "loss should decrease"
print("OK, final logged loss:", hist[-1][1])

step 5 | avg loss 2.570233


epoch 1/3 complete


step 10 | avg loss 2.293599


step 15 | avg loss 2.279374


epoch 2/3 complete


step 20 | avg loss 2.093139


epoch 3/3 complete


OK, final logged loss: 2.0931393265724183
